# Analyzing the distance covered and anomalities

In [1]:
import duckdb

In [2]:
df = duckdb.read_parquet('../raw/*/*.parquet')
df

VendorIDBIGINT,tpep_pickup_datetimeTIMESTAMP,tpep_dropoff_datetimeTIMESTAMP,passenger_countDOUBLE,trip_distanceDOUBLE,RatecodeIDDOUBLE,store_and_fwd_flagVARCHAR,PULocationIDBIGINT,DOLocationIDBIGINT,payment_typeBIGINT,fare_amountDOUBLE,extraDOUBLE,mta_taxDOUBLE,tip_amountDOUBLE,tolls_amountDOUBLE,improvement_surchargeDOUBLE,total_amountDOUBLE,congestion_surchargeDOUBLE,airport_feeDOUBLE
1,2022-01-01 00:35:40,2022-01-01 00:53:29,2.0,3.8,1.0,N,142,236,1,14.5,3.0,0.5,3.65,0.0,0.3,21.95,2.5,0.0
1,2022-01-01 00:33:43,2022-01-01 00:42:07,1.0,2.1,1.0,N,236,42,1,8.0,0.5,0.5,4.0,0.0,0.3,13.3,0.0,0.0
2,2022-01-01 00:53:21,2022-01-01 01:02:19,1.0,0.97,1.0,N,166,166,1,7.5,0.5,0.5,1.76,0.0,0.3,10.56,0.0,0.0
2,2022-01-01 00:25:21,2022-01-01 00:35:23,1.0,1.09,1.0,N,114,68,2,8.0,0.5,0.5,0.0,0.0,0.3,11.8,2.5,0.0
2,2022-01-01 00:36:48,2022-01-01 01:14:20,1.0,4.3,1.0,N,68,163,1,23.5,0.5,0.5,3.0,0.0,0.3,30.3,2.5,0.0
1,2022-01-01 00:40:15,2022-01-01 01:09:48,1.0,10.3,1.0,N,138,161,1,33.0,3.0,0.5,13.0,6.55,0.3,56.35,2.5,0.0
2,2022-01-01 00:20:50,2022-01-01 00:34:58,1.0,5.07,1.0,N,233,87,1,17.0,0.5,0.5,5.2,0.0,0.3,26.0,2.5,0.0
2,2022-01-01 00:13:04,2022-01-01 00:22:45,1.0,2.02,1.0,N,238,152,2,9.0,0.5,0.5,0.0,0.0,0.3,12.8,2.5,0.0
2,2022-01-01 00:30:02,2022-01-01 00:44:49,1.0,2.71,1.0,N,166,236,1,12.0,0.5,0.5,2.25,0.0,0.3,18.05,2.5,0.0
2,2022-01-01 00:48:52,2022-01-01 00:53:28,1.0,0.78,1.0,N,236,141,2,5.0,0.5,0.5,0.0,0.0,0.3,8.8,2.5,0.0


## Filtering out extreme outliers (Distance)

In [10]:
zero = duckdb.sql("""
                    SELECT
                        trip_distance,
                        fare_amount,
                        total_amount
                    FROM 
                        df
                    WHERE
                        trip_distance ==  0
                    """).df()
zero

,trip_distance,fare_amount,total_amount
0,0.0,2.50,6.30
1,0.0,2.50,6.30
2,0.0,2.50,6.30
3,0.0,2.50,6.30
4,0.0,10.00,15.36
...,...,...,...
3526774,0.0,22.81,27.95
3526775,0.0,50.66,55.41
3526776,0.0,27.50,43.12
3526777,0.0,9.19,13.94


**There are over `3.5M` rows where distance travelled is 0:**
* For some of them faires are charged suggesting it might be a cancelled ride
* It could be useful for revenue analysis

In [11]:
td_l_0 = duckdb.sql("""
                    SELECT
                        trip_distance,
                        fare_amount,
                        total_amount
                    FROM 
                        df
                    WHERE
                        trip_distance > 500
                    """).df()
td_l_0

,trip_distance,fare_amount,total_amount
0,577.50,52.00,71.13
1,622.00,52.00,75.70
2,572.60,52.00,61.85
3,616.80,52.00,75.70
4,651.00,52.00,71.85
...,...,...,...
5147,95310.01,86.69,107.39
5148,92140.64,46.82,48.32
5149,139278.22,24.75,29.50
5150,221055.79,29.93,38.14


In [12]:
td_l_0.describe()

,trip_distance,fare_amount,total_amount
count,5152.000000,5152.000000,5152.000000
mean,69017.704606,32.847125,39.677937
std,61853.257809,645.163694,645.268910
min,500.640000,-633.110000,-636.360000
25%,19448.175000,12.627500,17.587500
50%,58141.365000,19.895000,25.195000
75%,99157.192500,31.440000,38.177500
max,398608.620000,46263.880000,46269.440000


**From the above analysis it is clear that there are some anomalities in the trip distance column**

*For example:*
* Earth's Circumference: `24,901.461 mi`
* The maximum trip distance recorded is:`398608.62 mi`
    * Therefore the maximum noted trip distance is `~16 times` earth's circumference

**There are about 5152 rows which contain distances greater than 500mi** 
* Removing those data to get a cleaner dataset

### Filtering out rows where trip_distance is 0 mi and greater than 500 mi

In [13]:
df1 = duckdb.sql("""
                    SELECT
                        *
                    FROM 
                        df
                    WHERE
                        trip_distance < 500 
                    AND
                       trip_distance <> 0 
                    """)
df1

VendorIDBIGINT,tpep_pickup_datetimeTIMESTAMP,tpep_dropoff_datetimeTIMESTAMP,passenger_countDOUBLE,trip_distanceDOUBLE,RatecodeIDDOUBLE,store_and_fwd_flagVARCHAR,PULocationIDBIGINT,DOLocationIDBIGINT,payment_typeBIGINT,fare_amountDOUBLE,extraDOUBLE,mta_taxDOUBLE,tip_amountDOUBLE,tolls_amountDOUBLE,improvement_surchargeDOUBLE,total_amountDOUBLE,congestion_surchargeDOUBLE,airport_feeDOUBLE
1,2022-01-01 00:35:40,2022-01-01 00:53:29,2.0,3.8,1.0,N,142,236,1,14.5,3.0,0.5,3.65,0.0,0.3,21.95,2.5,0.0
1,2022-01-01 00:33:43,2022-01-01 00:42:07,1.0,2.1,1.0,N,236,42,1,8.0,0.5,0.5,4.0,0.0,0.3,13.3,0.0,0.0
2,2022-01-01 00:53:21,2022-01-01 01:02:19,1.0,0.97,1.0,N,166,166,1,7.5,0.5,0.5,1.76,0.0,0.3,10.56,0.0,0.0
2,2022-01-01 00:25:21,2022-01-01 00:35:23,1.0,1.09,1.0,N,114,68,2,8.0,0.5,0.5,0.0,0.0,0.3,11.8,2.5,0.0
2,2022-01-01 00:36:48,2022-01-01 01:14:20,1.0,4.3,1.0,N,68,163,1,23.5,0.5,0.5,3.0,0.0,0.3,30.3,2.5,0.0
1,2022-01-01 00:40:15,2022-01-01 01:09:48,1.0,10.3,1.0,N,138,161,1,33.0,3.0,0.5,13.0,6.55,0.3,56.35,2.5,0.0
2,2022-01-01 00:20:50,2022-01-01 00:34:58,1.0,5.07,1.0,N,233,87,1,17.0,0.5,0.5,5.2,0.0,0.3,26.0,2.5,0.0
2,2022-01-01 00:13:04,2022-01-01 00:22:45,1.0,2.02,1.0,N,238,152,2,9.0,0.5,0.5,0.0,0.0,0.3,12.8,2.5,0.0
2,2022-01-01 00:30:02,2022-01-01 00:44:49,1.0,2.71,1.0,N,166,236,1,12.0,0.5,0.5,2.25,0.0,0.3,18.05,2.5,0.0
2,2022-01-01 00:48:52,2022-01-01 00:53:28,1.0,0.78,1.0,N,236,141,2,5.0,0.5,0.5,0.0,0.0,0.3,8.8,2.5,0.0


In [14]:
duckdb.sql("""
            SELECT
                COUNT (*)
            FROM
                df1
            WHERE 
                trip_distance > 100
            """)

count_star()BIGINT
2164


**There are only 2164 rows with miles greater than 100**

* As 75% of the data falls with 3.5 mi range it is highly likely that the trip distance of more than 100mi are anomalities

### Filtering out rows where trip_distance is greater than 100

In [15]:
df1 = duckdb.sql("""
                    SELECT
                        *
                    FROM 
                        df
                    WHERE
                        trip_distance < 100
                    """)
df1

VendorIDBIGINT,tpep_pickup_datetimeTIMESTAMP,tpep_dropoff_datetimeTIMESTAMP,passenger_countDOUBLE,trip_distanceDOUBLE,RatecodeIDDOUBLE,store_and_fwd_flagVARCHAR,PULocationIDBIGINT,DOLocationIDBIGINT,payment_typeBIGINT,fare_amountDOUBLE,extraDOUBLE,mta_taxDOUBLE,tip_amountDOUBLE,tolls_amountDOUBLE,improvement_surchargeDOUBLE,total_amountDOUBLE,congestion_surchargeDOUBLE,airport_feeDOUBLE
1,2022-01-01 00:35:40,2022-01-01 00:53:29,2.0,3.8,1.0,N,142,236,1,14.5,3.0,0.5,3.65,0.0,0.3,21.95,2.5,0.0
1,2022-01-01 00:33:43,2022-01-01 00:42:07,1.0,2.1,1.0,N,236,42,1,8.0,0.5,0.5,4.0,0.0,0.3,13.3,0.0,0.0
2,2022-01-01 00:53:21,2022-01-01 01:02:19,1.0,0.97,1.0,N,166,166,1,7.5,0.5,0.5,1.76,0.0,0.3,10.56,0.0,0.0
2,2022-01-01 00:25:21,2022-01-01 00:35:23,1.0,1.09,1.0,N,114,68,2,8.0,0.5,0.5,0.0,0.0,0.3,11.8,2.5,0.0
2,2022-01-01 00:36:48,2022-01-01 01:14:20,1.0,4.3,1.0,N,68,163,1,23.5,0.5,0.5,3.0,0.0,0.3,30.3,2.5,0.0
1,2022-01-01 00:40:15,2022-01-01 01:09:48,1.0,10.3,1.0,N,138,161,1,33.0,3.0,0.5,13.0,6.55,0.3,56.35,2.5,0.0
2,2022-01-01 00:20:50,2022-01-01 00:34:58,1.0,5.07,1.0,N,233,87,1,17.0,0.5,0.5,5.2,0.0,0.3,26.0,2.5,0.0
2,2022-01-01 00:13:04,2022-01-01 00:22:45,1.0,2.02,1.0,N,238,152,2,9.0,0.5,0.5,0.0,0.0,0.3,12.8,2.5,0.0
2,2022-01-01 00:30:02,2022-01-01 00:44:49,1.0,2.71,1.0,N,166,236,1,12.0,0.5,0.5,2.25,0.0,0.3,18.05,2.5,0.0
2,2022-01-01 00:48:52,2022-01-01 00:53:28,1.0,0.78,1.0,N,236,141,2,5.0,0.5,0.5,0.0,0.0,0.3,8.8,2.5,0.0


In [16]:
trip_tw = duckdb.sql("""
                SELECT 
                    trip_distance
                FROM df1
                ORDER BY trip_distance DESC
                """).df()

trip_tw

,trip_distance
0,99.92
1,99.91
2,99.91
3,99.90
4,99.89
...,...
167851325,0.00
167851326,0.00
167851327,0.00
167851328,0.00


In [17]:
trip_tw.describe()

,trip_distance
count,1.678513e+08
mean,3.427596e+00
std,4.423333e+00
min,0.000000e+00
25%,1.050000e+00
50%,1.810000e+00
75%,3.500000e+00
max,9.992000e+01


## Extracting Trip Distance Time

In [18]:
trip_dt = duckdb.sql("""
                 SELECT 
                 *,
                     EXTRACT(EPOCH FROM (
                         tpep_dropoff_datetime - tpep_pickup_datetime
                         )) / 60 AS trip_duration_minutes,
                     fare_amount,
                     total_amount
                 FROM df1
                 ORDER BY trip_distance DESC
                """)

trip_dt

VendorIDBIGINT,tpep_pickup_datetimeTIMESTAMP,tpep_dropoff_datetimeTIMESTAMP,passenger_countDOUBLE,trip_distanceDOUBLE,RatecodeIDDOUBLE,store_and_fwd_flagVARCHAR,PULocationIDBIGINT,DOLocationIDBIGINT,payment_typeBIGINT,fare_amountDOUBLE,extraDOUBLE,mta_taxDOUBLE,tip_amountDOUBLE,tolls_amountDOUBLE,improvement_surchargeDOUBLE,total_amountDOUBLE,congestion_surchargeDOUBLE,airport_feeDOUBLE,trip_duration_minutesDOUBLE,fare_amountDOUBLE,total_amountDOUBLE
2,2023-05-13 16:12:25,2023-05-13 18:55:01,1.0,99.92,5.0,N,132,265,2,400.0,0.0,0.0,0.0,21.3,1.0,424.05,0.0,1.75,162.6,400.0,424.05
2,2023-12-19 08:45:48,2023-12-19 10:32:48,1.0,99.91,5.0,N,170,265,4,600.0,0.0,0.0,0.0,0.0,1.0,601.0,0.0,0.0,107.0,600.0,601.0
2,2023-12-19 08:45:48,2023-12-19 10:32:48,1.0,99.91,5.0,N,170,265,4,-600.0,0.0,0.0,120.2,0.0,-1.0,-480.8,0.0,0.0,107.0,-600.0,-480.8
1,2023-11-04 22:07:51,2023-11-05 00:28:03,0.0,99.9,1.0,N,237,265,1,364.2,3.5,0.5,50.0,12.75,1.0,431.95,2.5,0.0,140.2,364.2,431.95
2,2022-10-27 20:08:37,2022-10-27 22:07:33,1.0,99.89,5.0,N,68,265,1,450.0,0.0,0.0,0.0,11.75,0.3,462.05,0.0,0.0,118.93333333333334,450.0,462.05
2,2024-10-27 01:28:18,2024-10-27 03:25:43,1.0,99.86,5.0,N,170,265,1,150.0,0.0,0.0,32.88,13.38,1.0,197.26,0.0,0.0,117.41666666666667,150.0,197.26
2,2024-11-27 20:27:33,2024-11-27 22:56:27,2.0,99.85,5.0,N,231,265,1,318.0,0.0,0.0,67.88,20.38,1.0,407.26,0.0,0.0,148.9,318.0,407.26
2,2023-11-27 13:20:22,2023-11-27 14:28:53,1.0,99.84,4.0,N,138,265,2,303.3,5.0,0.0,0.0,6.94,1.0,317.99,0.0,1.75,68.51666666666667,303.3,317.99
2,2023-11-27 13:20:22,2023-11-27 14:28:53,1.0,99.84,4.0,N,138,265,2,-303.3,-5.0,0.0,0.0,-6.94,-1.0,-317.99,0.0,-1.75,68.51666666666667,-303.3,-317.99
2,2025-05-31 10:28:00,2025-05-31 12:33:35,1.0,99.82,1.0,N,216,265,2,105.9,0.0,0.0,0.0,16.06,1.0,122.96,0.0,0.0,125.58333333333333,105.9,122.96


In [19]:
duckdb.sql("""
            SUMMARIZE
            SELECT 
                trip_duration_minutes
            FROM 
                trip_dt
            """)

column_nameVARCHAR,column_typeVARCHAR,minVARCHAR,maxVARCHAR,approx_uniqueBIGINT,avgVARCHAR,stdVARCHAR,q25VARCHAR,q50VARCHAR,q75VARCHAR,countBIGINT,"null_percentageDECIMAL(9,2)"
trip_duration_minutes,DOUBLE,-28248298.466666665,10322055.183333334,44727,25.77406790652053,9642.85832276292,7.771739328054175,12.894717072256645,20.85471911636941,167851330,0.00


### Analyzing drip duration minutes where it is less than or equal to zero

In [20]:
ztn_dt = duckdb.sql("""
            SELECT 
                *
            FROM 
                trip_dt
            WHERE
                trip_duration_minutes <= 0
        
      
                  """)
ztn_dt

VendorIDBIGINT,tpep_pickup_datetimeTIMESTAMP,tpep_dropoff_datetimeTIMESTAMP,passenger_countDOUBLE,trip_distanceDOUBLE,RatecodeIDDOUBLE,store_and_fwd_flagVARCHAR,PULocationIDBIGINT,DOLocationIDBIGINT,payment_typeBIGINT,fare_amountDOUBLE,extraDOUBLE,mta_taxDOUBLE,tip_amountDOUBLE,tolls_amountDOUBLE,improvement_surchargeDOUBLE,total_amountDOUBLE,congestion_surchargeDOUBLE,airport_feeDOUBLE,trip_duration_minutesDOUBLE,fare_amount_1DOUBLE,total_amount_1DOUBLE
7,2025-06-04 22:34:52,2025-06-04 22:34:52,1.0,99.06,4.0,N,237,265,1,194.1,0.0,0.5,59.95,0.0,1.0,259.8,2.5,0.0,0.0,194.1,259.8
7,2025-07-04 23:23:56,2025-07-04 23:23:56,2.0,98.39,4.0,N,138,265,1,63.9,0.0,0.5,14.63,0.0,1.0,87.78,0.0,6.75,0.0,63.9,87.78
7,2025-05-01 08:30:14,2025-05-01 08:30:14,1.0,97.71,4.0,N,230,38,2,109.4,0.0,0.5,0.0,6.94,1.0,121.09,2.5,0.0,0.0,109.4,121.09
7,2025-06-19 15:46:42,2025-06-19 15:46:42,1.0,95.32,4.0,N,138,265,2,199.0,0.0,0.5,0.0,8.68,1.0,215.93,0.0,6.75,0.0,199.0,215.93
7,2025-07-22 12:17:49,2025-07-22 12:17:49,2.0,95.24,1.0,N,138,260,3,57.6,0.0,0.5,0.0,0.0,1.0,65.85,0.0,6.75,0.0,57.6,65.85
7,2025-10-21 12:41:22,2025-10-21 12:41:22,1.0,95.11,4.0,N,132,265,1,160.5,0.0,0.5,53.29,13.88,1.0,230.92,0.0,1.75,0.0,160.5,230.92
7,2025-08-18 16:30:29,2025-08-18 16:30:29,1.0,94.32,5.0,N,132,265,2,2.9,0.0,0.5,0.0,6.94,1.0,13.09,0.0,1.75,0.0,2.9,13.09
7,2025-10-14 22:42:53,2025-10-14 22:42:53,1.0,92.83,4.0,N,138,265,1,201.1,0.0,0.5,43.46,6.94,1.0,260.75,0.0,6.75,0.0,201.1,260.75
7,2025-05-06 19:50:48,2025-05-06 19:50:48,1.0,92.59,4.0,N,132,200,1,132.5,0.0,0.5,10.0,6.94,1.0,155.19,0.0,1.75,0.0,132.5,155.19
7,2025-04-16 19:20:44,2025-04-16 19:20:44,1.0,91.65,4.0,N,138,265,1,164.0,0.0,0.5,36.34,6.94,1.0,218.03,0.0,6.75,0.0,164.0,218.03


In [21]:
duckdb.sql("""
            SELECT
                COUNT (*) AS negative_duration_count,

            FROM
                ztn_dt
            """)

negative_duration_countBIGINT
607220


**There are about 607220 rows containing zero/negative trip duration minutes**

### Dropping zero/negative trip duration minutes

In [22]:
trip_dt = duckdb.sql("""
                    SELECT 
                        *
                    FROM 
                        trip_dt
                    WHERE 
                        trip_duration_minutes > 0
                    """)
trip_dt

VendorIDBIGINT,tpep_pickup_datetimeTIMESTAMP,tpep_dropoff_datetimeTIMESTAMP,passenger_countDOUBLE,trip_distanceDOUBLE,RatecodeIDDOUBLE,store_and_fwd_flagVARCHAR,PULocationIDBIGINT,DOLocationIDBIGINT,payment_typeBIGINT,fare_amountDOUBLE,extraDOUBLE,mta_taxDOUBLE,tip_amountDOUBLE,tolls_amountDOUBLE,improvement_surchargeDOUBLE,total_amountDOUBLE,congestion_surchargeDOUBLE,airport_feeDOUBLE,trip_duration_minutesDOUBLE,fare_amount_1DOUBLE,total_amount_1DOUBLE
2,2023-05-13 16:12:25,2023-05-13 18:55:01,1.0,99.92,5.0,N,132,265,2,400.0,0.0,0.0,0.0,21.3,1.0,424.05,0.0,1.75,162.6,400.0,424.05
2,2023-12-19 08:45:48,2023-12-19 10:32:48,1.0,99.91,5.0,N,170,265,4,600.0,0.0,0.0,0.0,0.0,1.0,601.0,0.0,0.0,107.0,600.0,601.0
2,2023-12-19 08:45:48,2023-12-19 10:32:48,1.0,99.91,5.0,N,170,265,4,-600.0,0.0,0.0,120.2,0.0,-1.0,-480.8,0.0,0.0,107.0,-600.0,-480.8
1,2023-11-04 22:07:51,2023-11-05 00:28:03,0.0,99.9,1.0,N,237,265,1,364.2,3.5,0.5,50.0,12.75,1.0,431.95,2.5,0.0,140.2,364.2,431.95
2,2022-10-27 20:08:37,2022-10-27 22:07:33,1.0,99.89,5.0,N,68,265,1,450.0,0.0,0.0,0.0,11.75,0.3,462.05,0.0,0.0,118.93333333333334,450.0,462.05
2,2024-10-27 01:28:18,2024-10-27 03:25:43,1.0,99.86,5.0,N,170,265,1,150.0,0.0,0.0,32.88,13.38,1.0,197.26,0.0,0.0,117.41666666666667,150.0,197.26
2,2024-11-27 20:27:33,2024-11-27 22:56:27,2.0,99.85,5.0,N,231,265,1,318.0,0.0,0.0,67.88,20.38,1.0,407.26,0.0,0.0,148.9,318.0,407.26
2,2023-11-27 13:20:22,2023-11-27 14:28:53,1.0,99.84,4.0,N,138,265,2,-303.3,-5.0,0.0,0.0,-6.94,-1.0,-317.99,0.0,-1.75,68.51666666666667,-303.3,-317.99
2,2023-11-27 13:20:22,2023-11-27 14:28:53,1.0,99.84,4.0,N,138,265,2,303.3,5.0,0.0,0.0,6.94,1.0,317.99,0.0,1.75,68.51666666666667,303.3,317.99
2,2025-05-31 10:28:00,2025-05-31 12:33:35,1.0,99.82,1.0,N,216,265,2,105.9,0.0,0.0,0.0,16.06,1.0,122.96,0.0,0.0,125.58333333333333,105.9,122.96
